# Bài 03 — Ma trận đơn vị, nghịch đảo và ma trận đặc biệt với NumPy

Notebook này đi từ cấu trúc ma trận đến cách giải hệ tuyến tính và hồi quy. Mỗi phần đều yêu cầu bạn đọc **shape**, kiểm tra kết quả bằng **residual** và phân biệt một nghiệm tính được với một nghiệm ổn định.

## Mục tiêu

Sau khi hoàn thành, bạn có thể:

- tạo và nhận diện identity, diagonal, triangular, symmetric, permutation matrix;
- kiểm chứng nghịch đảo bằng sai số số học;
- dùng `solve` cho một hoặc nhiều vế phải;
- phát hiện hệ suy biến/gần suy biến bằng rank và condition number;
- dùng `lstsq` và `pinv` cho bài toán không vuông;
- fit linear regression và Ridge mà không tạo nghịch đảo tường minh.

> Cách học: dự đoán shape/kết quả trước, chạy cell, rồi giải thích assertion bằng lời.

## 0. Khởi tạo

In [1]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(42)

def show(name, value):
    array = np.asarray(value)
    print(f"{name}: shape={array.shape}, ndim={array.ndim}, dtype={array.dtype}")
    print(array, "\n")

def residual_norm(A, x, b):
    return np.linalg.norm(A @ x - b)

print("NumPy:", np.__version__)

NumPy: 2.4.3


## 1. Ma trận đơn vị và hợp đồng shape

Với `A.shape == (m, n)`, identity bên trái phải là `(m, m)`, còn identity bên phải phải là `(n, n)`.

In [2]:
A_rect = np.arange(1, 7, dtype=float).reshape(2, 3)
I_left = np.eye(A_rect.shape[0])
I_right = np.eye(A_rect.shape[1])

show("A_rect", A_rect)
show("I_left", I_left)
show("I_right", I_right)

np.testing.assert_allclose(I_left @ A_rect, A_rect)
np.testing.assert_allclose(A_rect @ I_right, A_rect)

A_rect: shape=(2, 3), ndim=2, dtype=float64
[[1. 2. 3.]
 [4. 5. 6.]] 

I_left: shape=(2, 2), ndim=2, dtype=float64
[[1. 0.]
 [0. 1.]] 

I_right: shape=(3, 3), ndim=2, dtype=float64
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]] 



## 2. Ma trận chéo: tạo, trích xuất và scale feature

`np.diag` có hai vai trò: vector → ma trận chéo và ma trận → vector đường chéo. Với dữ liệu lớn, broadcasting thường tốt hơn tạo ma trận chéo dày đặc.

In [3]:
scales = np.array([2.0, 0.5, -1.0])
D = np.diag(scales)
X_features = np.array([[1.0, 10.0, 100.0], [2.0, 20.0, 200.0]])

scaled_by_matrix = X_features @ D
scaled_by_broadcast = X_features * scales

show("D", D)
show("X @ D", scaled_by_matrix)
np.testing.assert_array_equal(np.diag(D), scales)
np.testing.assert_allclose(scaled_by_matrix, scaled_by_broadcast)

D_inv = np.diag(1.0 / scales)
np.testing.assert_allclose(D @ D_inv, np.eye(3))

D: shape=(3, 3), ndim=2, dtype=float64
[[ 2.   0.   0. ]
 [ 0.   0.5  0. ]
 [ 0.   0.  -1. ]] 

X @ D: shape=(2, 3), ndim=2, dtype=float64
[[   2.    5. -100.]
 [   4.   10. -200.]] 



## 3. Ma trận tam giác và đối xứng

`triu` giữ phần tam giác trên; `tril` giữ phần tam giác dưới. Ma trận `M.T @ M` luôn đối xứng, dù `M` có vuông hay không.

In [4]:
M = np.array([[2.0, -1.0, 3.0], [4.0, 0.5, 2.0], [-2.0, 1.0, 5.0]])
upper = np.triu(M)
lower = np.tril(M)
symmetric_part = (M + M.T) / 2.0
gram = M.T @ M

show("upper", upper)
show("lower", lower)
show("M.T @ M", gram)
assert np.allclose(upper, np.triu(upper))
assert np.allclose(lower, np.tril(lower))
assert np.allclose(symmetric_part, symmetric_part.T)
assert np.allclose(gram, gram.T)

upper: shape=(3, 3), ndim=2, dtype=float64
[[ 2.  -1.   3. ]
 [ 0.   0.5  2. ]
 [ 0.   0.   5. ]] 

lower: shape=(3, 3), ndim=2, dtype=float64
[[ 2.   0.   0. ]
 [ 4.   0.5  0. ]
 [-2.   1.   5. ]] 

M.T @ M: shape=(3, 3), ndim=2, dtype=float64
[[24.   -2.    4.  ]
 [-2.    2.25  3.  ]
 [ 4.    3.   38.  ]] 



## 4. Ma trận hoán vị: đổi hàng, đổi cột và shuffle

Nhân `P @ A` đổi hàng; nhân `A @ P.T` dưới đây đổi cột theo cùng `order`. Trong thực tế, indexing rõ nghĩa và tiết kiệm hơn tạo `P`.

In [5]:
order = np.array([2, 0, 1])
P = np.eye(3)[order]
A_square = np.arange(1, 10).reshape(3, 3)

rows_by_P = P @ A_square
cols_by_P = A_square @ P.T

show("P", P)
show("P @ A", rows_by_P)
show("A @ P.T", cols_by_P)
np.testing.assert_allclose(rows_by_P, A_square[order])
np.testing.assert_allclose(cols_by_P, A_square[:, order])
np.testing.assert_allclose(P.T @ P, np.eye(3))
np.testing.assert_allclose(np.linalg.inv(P), P.T)

P: shape=(3, 3), ndim=2, dtype=float64
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]] 

P @ A: shape=(3, 3), ndim=2, dtype=float64
[[7. 8. 9.]
 [1. 2. 3.]
 [4. 5. 6.]] 

A @ P.T: shape=(3, 3), ndim=2, dtype=float64
[[3. 1. 2.]
 [6. 4. 5.]
 [9. 7. 8.]] 



In [6]:
X_pairs = np.array([[10, 100], [20, 200], [30, 300], [40, 400]])
y_pairs = np.array([1, 2, 3, 4])
shuffle_order = rng.permutation(len(X_pairs))
X_shuffled = X_pairs[shuffle_order]
y_shuffled = y_pairs[shuffle_order]

print("order:", shuffle_order)
print("pairs sau shuffle:")
print(np.column_stack([X_shuffled, y_shuffled]))
np.testing.assert_array_equal(X_shuffled[:, 0] // 10, y_shuffled)

order: [3 2 1 0]
pairs sau shuffle:
[[ 40 400   4]
 [ 30 300   3]
 [ 20 200   2]
 [ 10 100   1]]


## 5. Nghịch đảo là phép hoàn tác

Với số thực dấu phẩy động, kiểm tra `A @ A_inv ≈ I` bằng `allclose`, không dùng `==`.

In [7]:
A = np.array([[2.0, 1.0], [1.0, 1.0]])
A_inv = np.linalg.inv(A)
x_original = np.array([3.0, -2.0])
y_transformed = A @ x_original
x_recovered = A_inv @ y_transformed

show("A_inv", A_inv)
show("A @ A_inv", A @ A_inv)
np.testing.assert_allclose(A @ A_inv, np.eye(2), atol=1e-12)
np.testing.assert_allclose(A_inv @ A, np.eye(2), atol=1e-12)
np.testing.assert_allclose(x_recovered, x_original)

A_inv: shape=(2, 2), ndim=2, dtype=float64
[[ 1. -1.]
 [-1.  2.]] 

A @ A_inv: shape=(2, 2), ndim=2, dtype=float64
[[1. 0.]
 [0. 1.]] 



### Tính chất: nghịch đảo của tích đảo thứ tự

`inv(A @ B) == inv(B) @ inv(A)`, không phải `inv(A) @ inv(B)`.

In [8]:
B = np.array([[1.0, 2.0], [0.0, 1.0]])
inverse_product = np.linalg.inv(A @ B)
reverse_product = np.linalg.inv(B) @ np.linalg.inv(A)
wrong_order = np.linalg.inv(A) @ np.linalg.inv(B)

np.testing.assert_allclose(inverse_product, reverse_product)
assert not np.allclose(inverse_product, wrong_order)
print("max error nếu không đảo thứ tự:", np.max(np.abs(inverse_product - wrong_order)))

max error nếu không đảo thứ tự: 2.0


## 6. Giải hệ bằng `solve`, không bằng nghịch đảo

Hai cách dưới đây gần nhau về giá trị, nhưng `solve` là cách triển khai đúng mục đích. Luôn kiểm tra nghiệm bằng residual.

In [9]:
system_A = np.array([[3.0, 1.0], [1.0, 2.0]])
system_b = np.array([9.0, 8.0])

x_solve = np.linalg.solve(system_A, system_b)
x_inverse = np.linalg.inv(system_A) @ system_b

show("x_solve", x_solve)
print("residual norm:", residual_norm(system_A, x_solve, system_b))
np.testing.assert_allclose(x_solve, [2.0, 3.0])
np.testing.assert_allclose(x_solve, x_inverse)
assert residual_norm(system_A, x_solve, system_b) < 1e-12

x_solve: shape=(2,), ndim=1, dtype=float64
[2. 3.] 

residual norm: 0.0


### Nhiều vế phải trong một lần giải

Nếu `B.shape == (n, k)`, mỗi cột là một vế phải và kết quả có shape `(n, k)`.

In [10]:
right_sides = np.column_stack([system_b, np.array([1.0, 0.0]), np.array([0.0, 1.0])])
solutions = np.linalg.solve(system_A, right_sides)

show("right_sides", right_sides)
show("solutions", solutions)
assert solutions.shape == (2, 3)
np.testing.assert_allclose(system_A @ solutions, right_sides)
np.testing.assert_allclose(solutions[:, 0], x_solve)
np.testing.assert_allclose(solutions[:, 1:], np.linalg.inv(system_A))

right_sides: shape=(2, 3), ndim=2, dtype=float64
[[9. 1. 0.]
 [8. 0. 1.]] 

solutions: shape=(2, 3), ndim=2, dtype=float64
[[ 2.   0.4 -0.2]
 [ 3.  -0.2  0.6]] 



## 7. Ma trận suy biến: `solve` không thể cho nghiệm duy nhất

Hai hàng của `singular_A` phụ thuộc. Cell bắt lỗi có chủ đích để notebook tiếp tục chạy.

In [11]:
singular_A = np.array([[1.0, 2.0], [2.0, 4.0]])
consistent_b = np.array([3.0, 6.0])
inconsistent_b = np.array([3.0, 7.0])

print("rank:", np.linalg.matrix_rank(singular_A))
print("condition number:", np.linalg.cond(singular_A))
try:
    np.linalg.solve(singular_A, consistent_b)
except np.linalg.LinAlgError as error:
    print("Lỗi dự kiến từ solve:", error)

x_many, *_ = np.linalg.lstsq(singular_A, consistent_b, rcond=None)
x_approx, *_ = np.linalg.lstsq(singular_A, inconsistent_b, rcond=None)
print("một nghiệm cho hệ nhất quán:", x_many)
print("nghiệm xấp xỉ cho hệ mâu thuẫn:", x_approx)
print("hai residual norm:", residual_norm(singular_A, x_many, consistent_b), residual_norm(singular_A, x_approx, inconsistent_b))

rank: 1
condition number: 2.517588727560788e+16
Lỗi dự kiến từ solve: Singular matrix
một nghiệm cho hệ nhất quán: [0.6 1.2]
nghiệm xấp xỉ cho hệ mâu thuẫn: [0.68 1.36]
hai residual norm: 1.9860273225978185e-15 0.4472135954999579


## 8. Gần suy biến: residual nhỏ chưa đủ

Hai hệ chỉ khác vế phải một lượng rất nhỏ. Ma trận có condition number lớn sẽ khuếch đại thay đổi đó thành thay đổi lớn của nghiệm.

In [12]:
epsilon = 1e-8
near_A = np.array([[1.0, 1.0], [1.0, 1.0 + epsilon]])
b1 = np.array([2.0, 2.0 + epsilon])
b2 = b1 + np.array([0.0, epsilon])
x1 = np.linalg.solve(near_A, b1)
x2 = np.linalg.solve(near_A, b2)

print("cond(A):", np.linalg.cond(near_A))
print("||b2-b1||:", np.linalg.norm(b2 - b1))
print("x1:", x1, "x2:", x2, "||x2-x1||:", np.linalg.norm(x2 - x1))
print("residuals:", residual_norm(near_A, x1, b1), residual_norm(near_A, x2, b2))
assert np.linalg.cond(near_A) > 1e8
assert np.linalg.norm(x2 - x1) > 1.0

cond(A): 400000006.5548868
||b2-b1||: 9.99999993922529e-09
x1: [1. 1.] x2: [0. 2.] ||x2-x1||: 1.4142135623730951
residuals: 0.0 0.0


## 9. Least squares và giả nghịch đảo

Với `A.shape == (m, n)` và thường `m > n`, `lstsq` tìm `x` làm nhỏ nhất `||A @ x - b||²`. `pinv(A) @ b` cho cùng nghiệm cơ bản, nhưng `lstsq` thể hiện mục đích rõ hơn.

In [13]:
t = np.arange(5, dtype=float)
design = np.column_stack([np.ones_like(t), t])
targets = np.array([1.1, 2.9, 5.2, 6.8, 9.1])

coef_lstsq, residuals, rank, singular_values = np.linalg.lstsq(design, targets, rcond=None)
coef_pinv = np.linalg.pinv(design) @ targets
predictions = design @ coef_lstsq

print("coefficients [intercept, slope]:", coef_lstsq)
print("rank:", rank, "singular values:", singular_values)
print("residual norm:", np.linalg.norm(predictions - targets))
np.testing.assert_allclose(coef_lstsq, coef_pinv)
assert rank == design.shape[1]

coefficients [intercept, slope]: [1.04 1.99]
rank: 2 singular values: [5.78859 1.22155]
residual norm: 0.32710854467592276


## 10. Normal equation trong hồi quy tuyến tính

Ta so sánh ba cách trên ví dụ trong hình. Cách nghịch đảo chỉ để đối chiếu; trong code thực tế hãy ưu tiên `solve` hoặc `lstsq`.

In [14]:
reg_x = np.array([1.0, 2.0])
reg_y = np.array([3.0, 5.0])
reg_X = np.column_stack([np.ones_like(reg_x), reg_x])
reg_gram = reg_X.T @ reg_X
reg_rhs = reg_X.T @ reg_y

w_inverse = np.linalg.inv(reg_gram) @ reg_rhs
w_solve = np.linalg.solve(reg_gram, reg_rhs)
w_lstsq = np.linalg.lstsq(reg_X, reg_y, rcond=None)[0]

show("X.T @ X", reg_gram)
show("w", w_solve)
np.testing.assert_allclose(w_solve, [1.0, 2.0])
np.testing.assert_allclose(w_inverse, w_solve)
np.testing.assert_allclose(w_lstsq, w_solve)
np.testing.assert_allclose(reg_X @ w_solve, reg_y)

X.T @ X: shape=(2, 2), ndim=2, dtype=float64
[[2. 3.]
 [3. 5.]] 

w: shape=(2,), ndim=1, dtype=float64
[1. 2.] 



### `X.T @ X` làm độ điều kiện xấu hơn

Khi feature gần phụ thuộc, condition number của Gram matrix xấp xỉ bình phương condition number của `X` (theo norm 2). Đây là lý do `lstsq(X, y)` thường ổn định hơn tự lập normal equation.

In [15]:
base_feature = np.linspace(-1.0, 1.0, 30)
near_duplicate = base_feature + 1e-5 * rng.normal(size=base_feature.size)
X_near_collinear = np.column_stack([np.ones_like(base_feature), base_feature, near_duplicate])
cond_X = np.linalg.cond(X_near_collinear)
cond_gram = np.linalg.cond(X_near_collinear.T @ X_near_collinear)

print("cond(X):", cond_X)
print("cond(X.T @ X):", cond_gram)
print("cond(X)^2:", cond_X ** 2)
assert cond_gram > cond_X

cond(X): 174289.30589977594
cond(X.T @ X): 30376778906.014874
cond(X)^2: 30376762151.02567


## 11. Ridge: identity matrix giúp ổn định hệ

Ridge giải `(X.T @ X + λP) @ w = X.T @ y`. Ở đây `P[0, 0] = 0` để không phạt intercept.

In [16]:
ridge_y = 1.5 + 2.0 * base_feature + 0.1 * rng.normal(size=base_feature.size)
lambda_ = 0.1
penalty = np.eye(X_near_collinear.shape[1])
penalty[0, 0] = 0.0
ridge_system = X_near_collinear.T @ X_near_collinear + lambda_ * penalty
ridge_rhs = X_near_collinear.T @ ridge_y
w_ridge = np.linalg.solve(ridge_system, ridge_rhs)
w_unregularized = np.linalg.lstsq(X_near_collinear, ridge_y, rcond=None)[0]

print("unregularized:", w_unregularized)
print("ridge:", w_ridge)
print("cond Gram trước/sau Ridge:", cond_gram, np.linalg.cond(ridge_system))
assert np.linalg.cond(ridge_system) < cond_gram
assert np.linalg.norm(w_ridge[1:]) < np.linalg.norm(w_unregularized[1:])

unregularized: [    1.50213  3293.71674 -3291.72191]
ridge: [1.49886 0.9877  0.98764]
cond Gram trước/sau Ridge: 30376778906.014874 299.99999703803


## Bài tập

Có 7 bài tập tăng dần và một bài tổng hợp. Với mỗi bài:

1. dự đoán shape và viết công thức trước;
2. bỏ dấu `#` và hoàn thành `TODO`;
3. chạy ô bài làm rồi ô kiểm tra;
4. chỉ mở đáp án sau khi đã thử.

Ô kiểm tra chỉ nhắc biến còn thiếu, vì vậy notebook vẫn chạy từ đầu đến cuối khi chưa làm bài.

### Bài 1 — Identity đúng kích thước

Với `ex1_A.shape == (4, 2)`, tạo `ex1_left` và `ex1_right` để hai phép nhân identity đều giữ nguyên `ex1_A`.

In [17]:
ex1_A = np.arange(8, dtype=float).reshape(4, 2)

# TODO
# ex1_left = ...
# ex1_right = ...

In [18]:
if "ex1_left" not in globals() or "ex1_right" not in globals():
    print("⏳ Chưa làm Bài 1: hãy tạo ex1_left và ex1_right.")
else:
    assert ex1_left.shape == (4, 4)
    assert ex1_right.shape == (2, 2)
    np.testing.assert_allclose(ex1_left @ ex1_A, ex1_A)
    np.testing.assert_allclose(ex1_A @ ex1_right, ex1_A)
    print("✅ Bài 1 đúng")

⏳ Chưa làm Bài 1: hãy tạo ex1_left và ex1_right.


<details><summary>Đáp án Bài 1</summary>

```python
ex1_left = np.eye(ex1_A.shape[0])
ex1_right = np.eye(ex1_A.shape[1])
```
</details>

### Bài 2 — Scale feature không tạo ma trận chéo

Tạo `ex2_scaled` bằng broadcasting để scale từng cột của `ex2_X` theo `ex2_scales`. Không dùng `np.diag`.

In [19]:
ex2_X = np.arange(1, 13, dtype=float).reshape(4, 3)
ex2_scales = np.array([0.1, 10.0, -1.0])

# TODO
# ex2_scaled = ...

In [20]:
if "ex2_scaled" not in globals():
    print("⏳ Chưa làm Bài 2: hãy tạo ex2_scaled.")
else:
    assert ex2_scaled.shape == ex2_X.shape
    np.testing.assert_allclose(ex2_scaled, ex2_X @ np.diag(ex2_scales))
    print("✅ Bài 2 đúng")

⏳ Chưa làm Bài 2: hãy tạo ex2_scaled.


<details><summary>Đáp án Bài 2</summary>

```python
ex2_scaled = ex2_X * ex2_scales
```
</details>

### Bài 3 — Kiểm chứng nghịch đảo và residual

Tạo `ex3_inverse`, giải hệ bằng `solve` vào `ex3_x`, rồi tính `ex3_residual_norm`.

In [21]:
ex3_A = np.array([[4.0, 1.0], [2.0, 3.0]])
ex3_b = np.array([9.0, 13.0])

# TODO
# ex3_inverse = ...
# ex3_x = ...
# ex3_residual_norm = ...

In [22]:
ex3_names = ["ex3_inverse", "ex3_x", "ex3_residual_norm"]
if not all(name in globals() for name in ex3_names):
    print("⏳ Chưa làm Bài 3: hãy tạo", ex3_names)
else:
    assert ex3_inverse.shape == (2, 2)
    assert ex3_x.shape == (2,)
    np.testing.assert_allclose(ex3_A @ ex3_inverse, np.eye(2))
    np.testing.assert_allclose(ex3_A @ ex3_x, ex3_b)
    assert ex3_residual_norm < 1e-12
    print("✅ Bài 3 đúng")

⏳ Chưa làm Bài 3: hãy tạo ['ex3_inverse', 'ex3_x', 'ex3_residual_norm']


<details><summary>Đáp án Bài 3</summary>

```python
ex3_inverse = np.linalg.inv(ex3_A)
ex3_x = np.linalg.solve(ex3_A, ex3_b)
ex3_residual_norm = np.linalg.norm(ex3_A @ ex3_x - ex3_b)
```
</details>

### Bài 4 — Nhiều vế phải

Giải đồng thời ba vế phải trong `ex4_B`, lưu kết quả `(3, 3)` vào `ex4_X`. Không dùng vòng lặp.

In [23]:
ex4_A = np.array([[3.0, 1.0, 0.0], [1.0, 4.0, 1.0], [0.0, 1.0, 2.0]])
ex4_B = np.column_stack([np.array([1.0, 2.0, 3.0]), np.eye(3)[:, 0], np.eye(3)[:, 1]])

# TODO
# ex4_X = ...

In [24]:
if "ex4_X" not in globals():
    print("⏳ Chưa làm Bài 4: hãy tạo ex4_X.")
else:
    assert ex4_X.shape == (3, 3)
    np.testing.assert_allclose(ex4_A @ ex4_X, ex4_B)
    for column in range(ex4_B.shape[1]):
        np.testing.assert_allclose(ex4_X[:, column], np.linalg.solve(ex4_A, ex4_B[:, column]))
    print("✅ Bài 4 đúng")

⏳ Chưa làm Bài 4: hãy tạo ex4_X.


<details><summary>Đáp án Bài 4</summary>

```python
ex4_X = np.linalg.solve(ex4_A, ex4_B)
```
</details>

### Bài 5 — Shuffle giữ nguyên cặp feature–label

Tạo đúng một `ex5_order`, rồi dùng lại nó để tạo `ex5_X_shuffled` và `ex5_y_shuffled`.

In [25]:
ex5_X = np.column_stack([np.arange(10), np.arange(10) ** 2])
ex5_y = 100 + np.arange(10)

# TODO
# ex5_order = ...
# ex5_X_shuffled = ...
# ex5_y_shuffled = ...

In [26]:
ex5_names = ["ex5_order", "ex5_X_shuffled", "ex5_y_shuffled"]
if not all(name in globals() for name in ex5_names):
    print("⏳ Chưa làm Bài 5: hãy tạo", ex5_names)
else:
    np.testing.assert_array_equal(np.sort(ex5_order), np.arange(10))
    np.testing.assert_array_equal(ex5_X_shuffled, ex5_X[ex5_order])
    np.testing.assert_array_equal(ex5_y_shuffled, ex5_y[ex5_order])
    np.testing.assert_array_equal(ex5_X_shuffled[:, 0] + 100, ex5_y_shuffled)
    print("✅ Bài 5 đúng")

⏳ Chưa làm Bài 5: hãy tạo ['ex5_order', 'ex5_X_shuffled', 'ex5_y_shuffled']


<details><summary>Đáp án Bài 5</summary>

```python
ex5_order = rng.permutation(len(ex5_X))
ex5_X_shuffled = ex5_X[ex5_order]
ex5_y_shuffled = ex5_y[ex5_order]
```
</details>

### Bài 6 — Fit đường thẳng bằng `lstsq`

Tạo design matrix có cột intercept vào `ex6_design`, fit `ex6_coef` bằng `lstsq`, rồi tạo `ex6_predictions`.

In [27]:
ex6_x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
ex6_y = np.array([-3.1, -1.0, 1.2, 3.0, 5.1])

# TODO
# ex6_design = ...
# ex6_coef = ...
# ex6_predictions = ...

In [28]:
ex6_names = ["ex6_design", "ex6_coef", "ex6_predictions"]
if not all(name in globals() for name in ex6_names):
    print("⏳ Chưa làm Bài 6: hãy tạo", ex6_names)
else:
    assert ex6_design.shape == (5, 2)
    assert ex6_coef.shape == (2,)
    assert ex6_predictions.shape == (5,)
    np.testing.assert_allclose(ex6_design[:, 0], 1.0)
    np.testing.assert_allclose(ex6_coef, np.linalg.pinv(ex6_design) @ ex6_y)
    assert np.linalg.norm(ex6_predictions - ex6_y) < 0.3
    print("✅ Bài 6 đúng")

⏳ Chưa làm Bài 6: hãy tạo ['ex6_design', 'ex6_coef', 'ex6_predictions']


<details><summary>Đáp án Bài 6</summary>

```python
ex6_design = np.column_stack([np.ones_like(ex6_x), ex6_x])
ex6_coef = np.linalg.lstsq(ex6_design, ex6_y, rcond=None)[0]
ex6_predictions = ex6_design @ ex6_coef
```
</details>

### Bài 7 — Ridge không regularize intercept

Tạo `ex7_penalty` có đường chéo `[0, 1, 1]`, rồi giải hệ Ridge vào `ex7_w`.

In [29]:
ex7_X = X_near_collinear.copy()
ex7_y = ridge_y.copy()
ex7_lambda = 0.5

# TODO
# ex7_penalty = ...
# ex7_system = ...
# ex7_rhs = ...
# ex7_w = ...

In [30]:
ex7_names = ["ex7_penalty", "ex7_system", "ex7_rhs", "ex7_w"]
if not all(name in globals() for name in ex7_names):
    print("⏳ Chưa làm Bài 7: hãy tạo", ex7_names)
else:
    np.testing.assert_array_equal(np.diag(ex7_penalty), [0.0, 1.0, 1.0])
    np.testing.assert_allclose(ex7_system, ex7_X.T @ ex7_X + ex7_lambda * ex7_penalty)
    np.testing.assert_allclose(ex7_rhs, ex7_X.T @ ex7_y)
    np.testing.assert_allclose(ex7_system @ ex7_w, ex7_rhs)
    assert np.linalg.cond(ex7_system) < np.linalg.cond(ex7_X.T @ ex7_X)
    print("✅ Bài 7 đúng")

⏳ Chưa làm Bài 7: hãy tạo ['ex7_penalty', 'ex7_system', 'ex7_rhs', 'ex7_w']


<details><summary>Đáp án Bài 7</summary>

```python
ex7_penalty = np.eye(ex7_X.shape[1])
ex7_penalty[0, 0] = 0.0
ex7_system = ex7_X.T @ ex7_X + ex7_lambda * ex7_penalty
ex7_rhs = ex7_X.T @ ex7_y
ex7_w = np.linalg.solve(ex7_system, ex7_rhs)
```
</details>

## Bài tổng hợp — Hồi quy giá nhà ổn định với feature gần trùng

Dataset có ba feature: diện tích, số phòng và một bản đo diện tích gần trùng. Hãy:

1. tạo `cap_design` gồm intercept và ba feature;
2. fit `cap_w_lstsq` bằng `lstsq` và lấy `cap_rank`, `cap_singular_values`;
3. tạo penalty không phạt intercept và fit `cap_w_ridge` với `cap_lambda`;
4. tạo hai vector prediction và RMSE tương ứng;
5. tính `cap_cond_design`, `cap_cond_gram`, `cap_cond_ridge`.

Không dùng `inv`. Quan sát: training RMSE của Ridge có thể nhỉnh hơn một chút, nhưng hệ và trọng số ổn định hơn.

In [31]:
cap_area = np.array([45.0, 55.0, 60.0, 72.0, 85.0, 95.0, 110.0, 125.0])
cap_rooms = np.array([1.0, 2.0, 2.0, 3.0, 3.0, 4.0, 4.0, 5.0])
cap_area_duplicate = cap_area + np.array([0.01, -0.01, 0.02, -0.02, 0.01, 0.0, -0.01, 0.02])
cap_y = np.array([1.25, 1.55, 1.68, 2.05, 2.32, 2.65, 2.98, 3.42])
cap_lambda = 1.0

# TODO 1: design matrix (8, 4)
# cap_design = ...

# TODO 2: least squares và thông tin chẩn đoán
# cap_w_lstsq, _, cap_rank, cap_singular_values = ...

# TODO 3: Ridge, không phạt intercept
# cap_penalty = ...
# cap_ridge_system = ...
# cap_w_ridge = ...

# TODO 4: prediction và RMSE
# cap_pred_lstsq = ...
# cap_pred_ridge = ...
# cap_rmse_lstsq = ...
# cap_rmse_ridge = ...

# TODO 5: condition numbers
# cap_cond_design = ...
# cap_cond_gram = ...
# cap_cond_ridge = ...

In [32]:
cap_names = [
    "cap_design", "cap_w_lstsq", "cap_rank", "cap_singular_values",
    "cap_penalty", "cap_ridge_system", "cap_w_ridge",
    "cap_pred_lstsq", "cap_pred_ridge", "cap_rmse_lstsq", "cap_rmse_ridge",
    "cap_cond_design", "cap_cond_gram", "cap_cond_ridge",
]
if not all(name in globals() for name in cap_names):
    missing = [name for name in cap_names if name not in globals()]
    print("⏳ Bài tổng hợp chưa đủ. Còn thiếu:", missing)
else:
    assert cap_design.shape == (8, 4)
    assert cap_w_lstsq.shape == (4,) and cap_w_ridge.shape == (4,)
    assert cap_pred_lstsq.shape == (8,) and cap_pred_ridge.shape == (8,)
    np.testing.assert_allclose(cap_design[:, 0], 1.0)
    np.testing.assert_array_equal(np.diag(cap_penalty), [0.0, 1.0, 1.0, 1.0])
    np.testing.assert_allclose(cap_ridge_system @ cap_w_ridge, cap_design.T @ cap_y)
    assert cap_rank == 4
    assert cap_cond_gram > cap_cond_design
    assert cap_cond_ridge < cap_cond_gram
    assert cap_rmse_lstsq <= cap_rmse_ridge + 1e-12
    assert np.linalg.norm(cap_w_ridge[1:]) < np.linalg.norm(cap_w_lstsq[1:])
    print("✅ Bài tổng hợp đúng")
    print("rank:", cap_rank, "singular values:", cap_singular_values)
    print("condition design / Gram / Ridge:", cap_cond_design, cap_cond_gram, cap_cond_ridge)
    print("RMSE least squares / Ridge:", cap_rmse_lstsq, cap_rmse_ridge)
    print("weights least squares:", cap_w_lstsq)
    print("weights Ridge:", cap_w_ridge)

⏳ Bài tổng hợp chưa đủ. Còn thiếu: ['cap_design', 'cap_w_lstsq', 'cap_rank', 'cap_singular_values', 'cap_penalty', 'cap_ridge_system', 'cap_w_ridge', 'cap_pred_lstsq', 'cap_pred_ridge', 'cap_rmse_lstsq', 'cap_rmse_ridge', 'cap_cond_design', 'cap_cond_gram', 'cap_cond_ridge']


<details><summary>Đáp án bài tổng hợp</summary>

```python
cap_design = np.column_stack([
    np.ones_like(cap_area), cap_area, cap_rooms, cap_area_duplicate
])

cap_w_lstsq, _, cap_rank, cap_singular_values = np.linalg.lstsq(
    cap_design, cap_y, rcond=None
)

cap_penalty = np.eye(cap_design.shape[1])
cap_penalty[0, 0] = 0.0
cap_ridge_system = cap_design.T @ cap_design + cap_lambda * cap_penalty
cap_w_ridge = np.linalg.solve(cap_ridge_system, cap_design.T @ cap_y)

cap_pred_lstsq = cap_design @ cap_w_lstsq
cap_pred_ridge = cap_design @ cap_w_ridge
cap_rmse_lstsq = np.sqrt(np.mean((cap_pred_lstsq - cap_y) ** 2))
cap_rmse_ridge = np.sqrt(np.mean((cap_pred_ridge - cap_y) ** 2))

cap_cond_design = np.linalg.cond(cap_design)
cap_cond_gram = np.linalg.cond(cap_design.T @ cap_design)
cap_cond_ridge = np.linalg.cond(cap_ridge_system)
```

Hai cột diện tích gần trùng làm hệ số least-squares riêng lẻ lớn và nhạy. Ridge không xóa thông tin, nhưng làm hệ dễ giải hơn và thu nhỏ norm của các hệ số.
</details>

## Tự kiểm tra nhanh

Không chạy code, hãy trả lời:

1. Với `A.shape == (5, 3)`, identity bên trái và bên phải có shape gì?
2. Tại sao `A @ inv(A)` nên được kiểm tra bằng `allclose`?
3. Vì sao `(A @ B)^-1` phải đảo thứ tự thành `B^-1 @ A^-1`?
4. `solve(A, B)` trả shape gì nếu `A` là `(6, 6)` và `B` là `(6, 4)`?
5. Residual nhỏ có đủ để kết luận nghiệm ổn định không?
6. Hai feature trùng nhau gây vấn đề gì cho `X.T @ X`?
7. Vì sao `lstsq(X, y)` thường tốt hơn tự giải normal equation?
8. `lambda * I` thay đổi hệ Ridge như thế nào?
9. Vì sao có thể đặt phần tử penalty của intercept bằng 0?
10. Tại sao shuffle `X` và `y` phải dùng cùng một permutation?

## Tổng kết

- Identity giữ nguyên phép biến đổi và tạo thành phần regularization trong Ridge.
- Ma trận chéo scale từng tọa độ; broadcasting thường tránh được ma trận dày đặc.
- Ma trận đối xứng, tam giác và hoán vị mang cấu trúc mà thuật toán có thể khai thác.
- Nghịch đảo mô tả phép hoàn tác, nhưng `solve(A, b)` là cách đúng để giải hệ vuông.
- Một ma trận suy biến không cho nghiệm duy nhất; ma trận gần suy biến cho nghiệm nhạy.
- Residual đo mức thỏa hệ, còn condition number đo độ nhạy—cần xem cả hai.
- `lstsq` giải hồi quy trực tiếp và thường tốt hơn tạo `X.T @ X`; `pinv` mở rộng khái niệm nghịch đảo.
- Ridge thêm penalty trên đường chéo, cải thiện độ ổn định và thu nhỏ hệ số.

Trước khi sang Bài 04, hãy tự tạo một hệ có residual nhỏ nhưng condition number lớn và giải thích vì sao hai thông tin đó không mâu thuẫn.